# Trajectory Tangling Analysis — topographic RNN

Replicates Amengual et al. tangling analysis (Russo et al., 2018 method)
on the topographic checkpoint (`stage2_topo.pt`, `BioLeakyRNNTopo`, 144 units).

**Paper anchors (FEF recordings):**
- Pre-target: tangling has inverted-U shape vs CTOA (quadratic R²=0.72, p=0.0017)
- Post-target: tangling decreases monotonically with CTOA (linear R²=0.97, p≈0)
- Post-target tangling correlates with RT: Spearman ρ=0.63, p<0.04
- Pre-target tangling correlates with position info: Spearman ρ=0.7, p<0.032


In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
from collections import defaultdict
import torch
import matplotlib.pyplot as plt

from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetWithDistractorsV3
from src.analysis import (
    collect_trials,
    tangling_by_ctoa_bin,
    polynomial_regression,
)
from src.plotting import (
    plot_tangling_timecourses,
    plot_tangling_vs_ctoa,
    plot_tangling_vs_rt,
)

device = "cpu"
print("device:", device)

from src.figsave import autosave
autosave("04b_tangling_topo")


## 1. Load model and collect trials

In [ ]:
from pathlib import Path

ckpt_path = Path("checkpoints/stage2_topo.pt")
assert ckpt_path.exists(), f"topo checkpoint not found: {ckpt_path}"


def make_model():
    return BioLeakyRNNTopo(
        input_size=7,
        hidden_size=180,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=0.10,
        rec_init="diag",
        use_ei=True,
        exc_ratio=0.80,
        use_dale=True,
        mask_seed=42,
        sheet_side=12,
        tau_ee=0.25,
        tau_ie=0.32,
        tau_ei=0.64,
        tau_ii=0.64,
        rf_sigma=0.3,
        tau_e_range=(50, 150),
        tau_i_range=(10, 50),
        use_adaptation=False,
        tau_adapt=100.0,
        g_adapt=0.5,
    )


def make_env():
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.6,
        distractor_strength=1.0,
        continuous_locations=True,
    )


model = make_model().to(device)
model.load_state_dict(
    torch.load(str(ckpt_path), map_location=device, weights_only=True)["state_dict"],
    strict=False,
)
model.eval()
model.noise_at_eval = True
print(f"Loaded {ckpt_path.name}, noise_at_eval={model.noise_at_eval}")

In [ ]:
print("Collecting 2000 trials...")
trials = collect_trials(model, make_env, n_trials=2000, device=device)

outcomes = {}
for tr in trials:
    outcomes[tr["train_outcome"]] = outcomes.get(tr["train_outcome"], 0) + 1
print("Outcomes:", outcomes)

## 2. Compute tangling

Windows aligned to target onset:
- **Pre-target**: −300..0 ms (steps −15..0)
- **Post-target**: 0..+600 ms (steps 0..+30)

PCA pre-reduction to 20D before computing tangling (reduces noise, speeds up computation).

In [ ]:
# Pre-target window: -300..0 ms
tang_pre = tangling_by_ctoa_bin(
    trials,
    align_key="target_onset",
    window_before=15,  # 300 ms
    window_after=0,
    pca_dims=20,
    outcome="correct",
    min_trials=10,
)

# Post-target window: 0..+600 ms
tang_post = tangling_by_ctoa_bin(
    trials,
    align_key="target_onset",
    window_before=0,
    window_after=30,  # 600 ms
    pca_dims=20,
    outcome="correct",
    min_trials=10,
)

print(
    f'Pre-target  bins: {tang_pre["labels"]}  Q_mean: {np.round(tang_pre["Q_mean"], 4)}'
)
print(
    f'Post-target bins: {tang_post["labels"]}  Q_mean: {np.round(tang_post["Q_mean"], 4)}'
)

## 3. Tangling time courses per CTOA bin

In [ ]:
# Combined pre + post window in one figure, matching paper Figure 7A
plot_tangling_timecourses(tang_pre, tang_post, align_label="target_onset")

## 4. Tangling vs CTOA regression

Paper:
- Pre-target: **quadratic** fit R²=0.72, p=0.0017 (inverted-U shape)
- Post-target: **linear** fit R²=0.97, p≈0 (monotone decrease)

In [ ]:
plot_tangling_vs_ctoa(tang_pre, tang_post)

In [ ]:
# Also compare linear vs quadratic for pre-target (AIC comparison)
x = tang_pre["ctoa_ms_mean"]
y = tang_pre["Q_mean"]

for deg in [1, 2]:
    reg = polynomial_regression(x, y, degree=deg)
    n = len(x)
    k = deg + 1  # number of parameters incl. intercept
    ss_res = (
        np.sum((reg["y"] - reg["y_hat"]) ** 2) if reg["y_hat"] is not None else np.nan
    )
    # AIC = n*log(ss_res/n) + 2k
    aic = n * np.log(ss_res / n) + 2 * k if np.isfinite(ss_res) and ss_res > 0 else np.nan
    print(
        f'Pre-target deg-{deg}: R²={reg["r2"]:.3f}  p={reg["p_value"]:.4f}  AIC={aic:.1f}'
    )

## 5. Post-target tangling vs RT

Paper: Spearman ρ=0.63, p<0.04 — longer CTOA → less tangling → faster RT.

In [ ]:
rt_by_bin = defaultdict(list)
for tr in trials:
    if tr["train_outcome"] == "correct" and tr.get("rt_ms") is not None:
        b = tr.get("ctoa_bin")
        if b is not None:
            rt_by_bin[b].append(tr["rt_ms"])

rt_mean = {b: np.mean(v) for b, v in rt_by_bin.items()}
print("Mean RT per CTOA bin (ms):")
for b in sorted(rt_mean):
    print(f"  bin {b}: {rt_mean[b]:.1f} ms  (n={len(rt_by_bin[b])})")

In [ ]:
fig, rho, p = plot_tangling_vs_rt(tang_post, rt_mean)

## 6. RT vs CTOA

Mean reaction time as a function of CTOA (correct trials only).

In [ ]:
# RT vs CTOA — linear regression + error bars (cf. paper Figure 5)
from scipy import stats as sp_stats
import numpy as np

bins = tang_post["labels"]
ctoa_x = tang_post["ctoa_ms_mean"]
rt_y = np.array([np.mean(rt_by_bin[b]) for b in bins])
rt_err = np.array([np.std(rt_by_bin[b], ddof=1) for b in bins])

slope, intercept, r, p_val, _ = sp_stats.linregress(ctoa_x, rt_y)
r2 = r**2
x_fit = np.linspace(ctoa_x.min(), ctoa_x.max(), 200)
y_fit = slope * x_fit + intercept

fig, ax = plt.subplots(figsize=(5, 4))
ax.errorbar(
    ctoa_x,
    rt_y,
    yerr=rt_err,
    fmt="s",
    color="steelblue",
    capsize=4,
    linewidth=1.5,
    markersize=6,
    label="mean +/- SD",
)
ax.plot(
    x_fit,
    y_fit,
    "--",
    color="firebrick",
    linewidth=1.5,
    label="R2=%.2f (p=%.3f)" % (r2, p_val),
)
ax.set_xlabel("CTOA (ms)", fontsize=12)
ax.set_ylabel("Reaction time (ms)", fontsize=12)
ax.set_title("RT vs CTOA", fontsize=12)
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
print("Linear regression: R2=%.3f  slope=%.4f ms/ms  p=%.4f" % (r2, slope, p_val))

## 7. Behavior vs CTOA (cf. Amengual Fig 2)

Hit rate = hits/(hits+misses); false-alarm rate = FA/(distractor-present trials);
RT = median per CTOA bin. Expectation grows with CTOA: hit rate up, FA rate up in
parallel, RT down.

In [ ]:
from collections import defaultdict
from scipy import stats as sp_stats

# Per-CTOA-bin behavioral metrics, computed directly from `trials`.
cnt = defaultdict(lambda: defaultdict(int)); ndist = defaultdict(int)
ctoa = defaultdict(list); rt = defaultdict(list)
for tr in trials:
    b = tr.get('ctoa_bin')
    if b is None:
        continue
    cnt[b][tr['train_outcome']] += 1
    hd = tr.get('has_distractors')
    hd = (tr.get('n_distractors', 0) > 0) if hd is None else hd
    if hd:
        ndist[b] += 1
    if tr.get('ctoa_ms') is not None:
        ctoa[b].append(tr['ctoa_ms'])
    if tr['train_outcome'] == 'correct' and tr.get('rt_ms') is not None:
        rt[b].append(tr['rt_ms'])

behav_bins = sorted(b for b in cnt if (cnt[b]['correct'] + cnt[b]['miss']) >= 5)
x_ctoa = np.array([np.mean(ctoa[b]) for b in behav_bins])
n_hm = np.array([cnt[b]['correct'] + cnt[b]['miss'] for b in behav_bins])
hit_rate = np.array([cnt[b]['correct'] / max(n, 1) for b, n in zip(behav_bins, n_hm)])
hit_se = np.sqrt(hit_rate * (1 - hit_rate) / np.clip(n_hm, 1, None))
n_d = np.array([max(ndist[b], 1) for b in behav_bins])
fa_rate = np.array([cnt[b].get('false_alarm', 0) / nd for b, nd in zip(behav_bins, n_d)])
fa_se = np.sqrt(fa_rate * (1 - fa_rate) / n_d)
rt_mean = np.array([np.mean(rt[b]) if rt[b] else np.nan for b in behav_bins])
rt_sd = np.array([np.std(rt[b], ddof=1) if len(rt[b]) > 1 else 0.0 for b in behav_bins])

def _fit(xx, yy):
    s, i, r, pv, _ = sp_stats.linregress(xx, yy); return s, i, r ** 2, pv

ok = ~np.isnan(rt_mean)
panels = [
    (hit_rate, hit_se, 'Hit rate', 'seagreen', _fit(x_ctoa, hit_rate), None, 'rate +/- binom SE'),
    (fa_rate, fa_se, 'False-alarm rate', 'darkorange', _fit(x_ctoa, fa_rate), None, 'rate +/- binom SE'),
    (rt_mean, rt_sd, 'Reaction time (ms)', 'steelblue', _fit(x_ctoa[ok], rt_mean[ok]), ok, 'mean +/- SD'),
]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (y, err, lab, col, fitres, msk, errlab) in zip(axes, panels):
    s, i, r2, pv = fitres
    xv = x_ctoa if msk is None else x_ctoa[msk]
    yv = y if msk is None else y[msk]
    ev = err if msk is None else err[msk]
    ax.errorbar(xv, yv, yerr=ev, fmt='s', color=col, capsize=4, lw=1.5, ms=6, label=errlab)
    xf = np.linspace(np.nanmin(x_ctoa), np.nanmax(x_ctoa), 100)
    ax.plot(xf, s * xf + i, '--', color='firebrick', lw=1.5, label='R2=%.2f (p=%.3f)' % (r2, pv))
    ax.set_xlabel('CTOA (ms)'); ax.set_title(lab); ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.suptitle('Behavior vs CTOA (cf. Amengual Fig 2)', fontsize=13)
fig.tight_layout()
fig.savefig('figures/behavior_vs_ctoa.png', dpi=140)
plt.show()

sh, ih, r2h, ph = _fit(x_ctoa, hit_rate)
sf, iff, r2f, pf = _fit(x_ctoa, fa_rate)
sr, ir, r2r, pr = _fit(x_ctoa[ok], rt_mean[ok])
shf, ihf, r2hf, phf = _fit(hit_rate, fa_rate)
print('Hit rate vs CTOA : R2=%.3f p=%.2e' % (r2h, ph))
print('FA rate vs CTOA  : R2=%.3f p=%.2e' % (r2f, pf))
print('RT vs CTOA       : R2=%.3f p=%.2e  slope=%.3f ms/ms' % (r2r, pr, sr))
print('Hit rate vs FA   : R2=%.3f p=%.2e' % (r2hf, phf))


## Summary

| Metric | Paper | Model |
|--------|-------|-------|
| Pre-target: quadratic R² | 0.72 | ? |
| Pre-target: quadratic p | 0.0017 | ? |
| Post-target: linear R² | 0.97 | ? |
| Post-target: linear p | ~0 | ? |
| Tangling vs RT: Spearman ρ | 0.63 | ? |
| Tangling vs RT: p | <0.04 | ? |

Fill in '?' from outputs above.

> **Note:** Tangling vs position info (Spearman ρ=0.7) requires decoding accuracy per CTOA bin — see `05_decoding.ipynb`.